# 

In [1]:
import jax
import jax.numpy as jnp

In [3]:
example_trees = [
    [1, 'a', object()],
    (1, (2, 3), ()),
    [1, {"k1": 2, "k2": (3, 4)}, 5],
    {"a": 2, "b": (2, 3)},
    jnp.array([1, 2, 3]),
]

In [4]:
for pytree in example_trees:
    leaves = jax.tree.leaves(pytree)
    print(f"{repr(pytree):<45} has {len(leaves)} leaves: {leaves}")

[1, 'a', <object object at 0x7f305ef80520>]   has 3 leaves: [1, 'a', <object object at 0x7f305ef80520>]
(1, (2, 3), ())                               has 3 leaves: [1, 2, 3]
[1, {'k1': 2, 'k2': (3, 4)}, 5]               has 5 leaves: [1, 2, 3, 4, 5]
{'a': 2, 'b': (2, 3)}                         has 3 leaves: [2, 2, 3]
Array([1, 2, 3], dtype=int32)                 has 1 leaves: [Array([1, 2, 3], dtype=int32)]


In [5]:
list_of_lists = [
    [1, 2, 3],
    [1, 2],
    [1, 2, 3, 4]
]

In [6]:
jax.tree.map(lambda x: x * 2, list_of_lists)

[[2, 4, 6], [2, 4], [2, 4, 6, 8]]

In [7]:
another_list_of_lists = list_of_lists
jax.tree.map(lambda x, y: x + y, list_of_lists, another_list_of_lists)

[[2, 4, 6], [2, 4], [2, 4, 6, 8]]

In [8]:
jax.tree.map(lambda x, y: x + y, list_of_lists, list_of_lists)

[[2, 4, 6], [2, 4], [2, 4, 6, 8]]

In [9]:
import numpy as np

In [10]:
def init_mlp_params(layer_widths):
    params = []
    for n_in, n_out in zip(layer_widths[:-1], layer_widths[1:]):
        params.append(
            dict(
                weights=np.random.normal(size=(n_in, n_out)) * np.sqrt(2 / n_in),
                biases=np.ones(shape=(n_out,))
            )
        )
    return params

params = init_mlp_params([1, 128, 128, 1])

In [11]:
jax.tree.map(lambda x: x.shape, params)

[{'biases': (128,), 'weights': (1, 128)},
 {'biases': (128,), 'weights': (128, 128)},
 {'biases': (1,), 'weights': (128, 1)}]

In [15]:
def forward(params, x):
    *hidden, last = params
    for layer in hidden:
        x = jax.nn.relu(x @ layer["weights"] + layer["biases"])
    return x @ last["weights"] + last["biases"]

In [17]:
# Mean squared error
def loss_fn(params, x, y):
    return jnp.mean((forward(params, x) - y) ** 2)

In [18]:
LEARNING_RATE = 0.0001

In [19]:
@jax.jit
def update(params, x, y):
    grads = jax.grad(loss_fn)(params, x, y)
    return jax.tree.map(
        lambda param, grad: param - LEARNING_RATE * grad, params, grads
    )

In [20]:
from jax.tree_util import tree_structure
print(tree_structure(object))

PyTreeDef(*)


In [21]:
import collections

ATuple = collections.namedtuple("ATuple", ('name'))

tree = [1, {'k1': 2, 'k2': (3, 4)}, ATuple('foo')]
flattened, _ = jax.tree_util.tree_flatten_with_path(tree)

for key_path, value in flattened:
  print(f'Value of tree{jax.tree_util.keystr(key_path)}: {value}')

Value of tree[0]: 1
Value of tree[1]['k1']: 2
Value of tree[1]['k2'][0]: 3
Value of tree[1]['k2'][1]: 4
Value of tree[2].name: foo


In [22]:
flattened

[((SequenceKey(idx=0),), 1),
 ((SequenceKey(idx=1), DictKey(key='k1')), 2),
 ((SequenceKey(idx=1), DictKey(key='k2'), SequenceKey(idx=0)), 3),
 ((SequenceKey(idx=1), DictKey(key='k2'), SequenceKey(idx=1)), 4),
 ((SequenceKey(idx=2), GetAttrKey(name='name')), 'foo')]

In [23]:
a_tree = [jnp.zeros((2, 3)), jnp.zeros((3, 4))]
a_tree

[Array([[0., 0., 0.],
        [0., 0., 0.]], dtype=float32),
 Array([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]], dtype=float32)]

In [24]:
shapes = jax.tree.map(lambda x: x.shape, a_tree)

In [25]:
shapes

[(2, 3), (3, 4)]

In [26]:
jax.tree.map(jnp.ones, shapes)

[(Array([1., 1.], dtype=float32), Array([1., 1., 1.], dtype=float32)),
 (Array([1., 1., 1.], dtype=float32), Array([1., 1., 1., 1.], dtype=float32))]

In [27]:
shapes2 = jax.tree.map(lambda x: jnp.array(x.shape), a_tree)
shapes2

[Array([2, 3], dtype=int32), Array([3, 4], dtype=int32)]

In [28]:
jax.tree.map(jnp.ones, shapes2)

[Array([[1., 1., 1.],
        [1., 1., 1.]], dtype=float32),
 Array([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]], dtype=float32)]

In [29]:
jax.tree.leaves([None, None, None])

[]

In [30]:
jax.tree.leaves([None, None, None], is_leaf=lambda x: x is None)

[None, None, None]

In [31]:
jax.tree.leaves([None, None, None, 1, 2, 3], is_leaf=lambda x: x is None)

[None, None, None, 1, 2, 3]

In [32]:
def tree_transpose(list_of_trees):
  """
  Converts a list of trees of identical structure into a single tree of lists.
  """
  return jax.tree.map(lambda *xs: list(xs), *list_of_trees)

# Convert a dataset from row-major to column-major.
episode_steps = [dict(t=1, obs=3), dict(t=2, obs=4)]
tree_transpose(episode_steps)

{'obs': [3, 4], 't': [1, 2]}

In [33]:
jax.tree.transpose(
  outer_treedef = jax.tree.structure([0 for e in episode_steps]),
  inner_treedef = jax.tree.structure(episode_steps[0]),
  pytree_to_transpose = episode_steps
)

{'obs': [3, 4], 't': [1, 2]}